# 06 · YOLO26x-seg vs BioTrove-CLIP + per-pixel head

Head-to-head on the **test split**, with identical metrics for both models:

| | YOLO26x-seg (`02_train`) | BioTrove-CLIP + head (`05_biotrove_semseg`) |
|---|---|---|
| Task | instance segmentation | semantic (per-pixel) segmentation |
| Backbone pretraining | COCO | ~40 M plant & animal photos (BioTrove) |
| Output | one mask per leaf / lesion | one class per pixel |

Metrics:
- **IoU per class** and mIoU (leaf + diseases) on per-pixel class maps (instance masks are painted into a class map: leaves first, diseases on top)
- **Coverage-% MAE** per disease at two levels: *image* (disease ÷ all leaf pixels, fair to both) and *leaf* (largest leaf, the backend rule; see the notes at the end)
- MAE on the images that **contain** the disease (most test images have no mildew, which makes the all-image MAE look small)
- **Lin's CCC**, **severity-class accuracy** (mild < 5 %, moderate 5–25 %, severe > 25 %), **Bland–Altman**
- Parameters and inference time per image
- **Bootstrap 95 % CIs** over test images and a **paired bootstrap test** for every metric

Results go to `results/`.

In [ ]:
import csv
import gc
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from dotenv import load_dotenv
from PIL import Image
from pycocotools.coco import COCO

from eval_utils import (CLASSES, DISEASES, confusion, count_params, coverage_primary_leaf, image_coverage,
                        iou_from_confusion, lins_ccc, load_predictor, paint_class_map, severity)

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
DATA_VERSION = f"v{os.environ['ROBOFLOW_VERSION']}"
DATA_NAME = f"{os.environ['ROBOFLOW_PROJECT']}-{DATA_VERSION}"
TEST_DIR = ROOT / "datasets" / f"{DATA_NAME}-coco" / "test"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

THRESHOLD = 0.5  # same confidence threshold as 03_evaluate
N_BOOT = 2000
SEED = 0
print("data:", DATA_NAME)

## The two models

Best checkpoint of each run. Set `RUNS` to the two run folders to compare: YOLO vs BioTrove, or the v1 YOLO vs the new YOLO trained on the clean `lds_v1` data. The test split always comes from `.env`, so both models are scored on the same images.

In [ ]:
YOLO_RUN = ROOT / "runs" / f"yolo26x-seg_data-{DATA_VERSION}"
BIOTROVE_RUN = ROOT / "runs" / f"semseg-biotrove-b16_data-{DATA_VERSION}"

# Run folders to compare (A, B); differences are reported as B − A. Both are scored on this .env's test split.
#   YOLO vs BioTrove on leaf-dis-seg v1:  {"YOLO26x-seg": YOLO_RUN, "BioTrove+head": BIOTROVE_RUN}
#   old vs new YOLO on the clean lds_v1 v2 test set:
RUNS = {
    "YOLO v1": ROOT / "runs" / "yolo26x-seg_data-v1",
    "YOLO lds-v2": ROOT / "runs" / "yolo26x-seg_data-v2",  # "yolo26x-seg_data-v2_lds" for a retrain
}


def best_checkpoint(run):
    return run / "weights" / "best.pt" if (run / "weights").exists() else run / "checkpoint_best.pth"


MODELS = {name: best_checkpoint(run) for name, run in RUNS.items()}
A, B = MODELS
for name, ckpt in MODELS.items():
    assert ckpt.exists(), f"{name}: {ckpt} not found; train it first"

# Training length, for the methods section
for name, run in RUNS.items():
    if (run / "results.csv").exists():  # YOLO
        with open(run / "results.csv") as f:
            rows = list(csv.DictReader(f))
        best = max(rows, key=lambda r: float(r["metrics/mAP50-95(M)"].strip()))
        print(f"{name:<14} {len(rows)} epochs trained, best val mask mAP50-95 {float(best['metrics/mAP50-95(M)']):.3f} "
              f"at epoch {int(float(best['epoch']))}")
    else:  # semseg
        with open(run / "history.csv") as f:
            rows = list(csv.DictReader(f))
        best = max(rows, key=lambda r: float(r["val_miou_fg"]))
        print(f"{name:<14} {len(rows)} epochs trained, best val mIoU(leaf+disease) {float(best['val_miou_fg']):.3f} "
              f"at epoch {best['epoch']}")

## Ground truth

Per test image: the class map, image-level coverage and primary-leaf coverage. Also counts how many test images contain each disease; with few positives the CIs for that disease will be wide.

In [ ]:
coco = COCO(str(TEST_DIR / "_annotations.coco.json"))
cat_names = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}
images = coco.loadImgs(coco.getImgIds())

gt = []
for img in images:
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img["id"]))
    masks = [coco.annToMask(a).astype(bool) for a in anns]
    names = [cat_names[a["category_id"]] for a in anns]
    cmap = paint_class_map(masks, names, (img["height"], img["width"]))
    gt.append({"map": cmap, "image": image_coverage(cmap), "leaf": coverage_primary_leaf(masks, names)})

print(len(gt), "test images")
for d in DISEASES:
    n_pos = sum(1 for g in gt if g["image"] and g["image"][d] > 0)
    print(f"  {d:<15} present in {n_pos} images")

# Example images for the figure: the two with the most mildew and the one with the most leaf spot
by = lambda d: sorted(range(len(gt)), key=lambda i: -(gt[i]["image"] or {d: 0})[d])
EXAMPLES = list(dict.fromkeys(by("powdery_mildew")[:2] + by("leaf_spot")[:1]))
print("example images:", [images[i]["file_name"][:40] for i in EXAMPLES])

## Predict with both models

Loaded one at a time and freed afterwards. Time per image = median over the test set (first image skipped as warm-up), including post-processing.

In [ ]:
per_image, meta, examples = {}, {}, {}

for name, ckpt in MODELS.items():
    kind, predict, model = load_predictor(ckpt, cat_names)
    recs, times = [], []
    for k, (img, g) in enumerate(zip(images, gt)):
        image = Image.open(TEST_DIR / img["file_name"]).convert("RGB")
        torch.cuda.synchronize(); t = time.perf_counter()
        masks, names, confidences = predict(image)
        torch.cuda.synchronize(); times.append(time.perf_counter() - t)
        keep = confidences >= THRESHOLD
        pred_map = paint_class_map(masks[keep], names[keep], g["map"].shape)
        recs.append({"conf": confusion(g["map"], pred_map), "image": image_coverage(pred_map),
                     "leaf": coverage_primary_leaf(masks[keep], names[keep])})
        if k in EXAMPLES:
            examples.setdefault(name, {})[k] = pred_map
    params = count_params(model)
    meta[name] = {"kind": kind, "params_M": round(params / 1e6, 1) if params else float("nan"),
                  "ms_per_image": round(1000 * float(np.median(times[1:])), 1)}
    per_image[name] = recs
    print(f"{name:<14} {meta[name]}")
    del model, predict
    gc.collect(); torch.cuda.empty_cache()

## Head-to-head table

Point estimate on the full test set, bootstrap 95 % CI, and the paired difference **BioTrove − YOLO** on the same resampled test sets (two-sided bootstrap p-value).
Higher is better for IoU / CCC / accuracy, lower for MAE. Coverage metrics use images where both ground truth and prediction contain a leaf; `missed leaf` counts images where the model found none.

In [ ]:
def metrics(recs, idx):
    iou = iou_from_confusion(sum(recs[i]["conf"] for i in idx))
    out = {f"iou_{c}": iou[c] for c in CLASSES[1:]}
    out["miou_fg"] = float(np.nanmean([iou[c] for c in CLASSES[1:]]))
    for level in ("image", "leaf"):
        pairs = [(gt[i][level], recs[i][level]) for i in idx if gt[i][level] is not None]
        found = [(g, p) for g, p in pairs if p is not None]
        out[f"{level}_missed_leaf"] = len(pairs) - len(found)
        for d in DISEASES:
            g = np.array([x[d] for x, _ in found]); p = np.array([y[d] for _, y in found])
            err = np.abs(p - g)
            out[f"{level}_mae_{d}"] = float(err.mean()) if len(g) else float("nan")
            out[f"{level}_mae_{d}_present"] = float(err[g > 0].mean()) if (g > 0).any() else float("nan")
            out[f"{level}_ccc_{d}"] = lins_ccc(g, p)
            out[f"{level}_sevacc_{d}"] = float(np.mean([severity(a) == severity(b) for a, b in zip(g, p)])) if len(g) else float("nan")
    return out


# (key, label, higher is better)
ROWS = [("miou_fg", "mIoU (leaf + diseases)", True),
        *[(f"iou_{c}", f"IoU {c}", True) for c in CLASSES[1:]]]
for level in ("image", "leaf"):
    for d in DISEASES:
        ROWS += [(f"{level}_mae_{d}", f"{level} MAE {d} (pp)", False),
                 (f"{level}_mae_{d}_present", f"{level} MAE {d}, images with it (pp)", False),
                 (f"{level}_ccc_{d}", f"{level} CCC {d}", True),
                 (f"{level}_sevacc_{d}", f"{level} severity acc {d}", True)]

n = len(gt)
rng = np.random.default_rng(SEED)
boot_idx = [rng.integers(0, n, n) for _ in range(N_BOOT)]  # same resamples for both models → paired test
point = {m: metrics(per_image[m], np.arange(n)) for m in MODELS}
boots = {m: [metrics(per_image[m], idx) for idx in boot_idx] for m in MODELS}


def ci(values):
    lo, hi = np.nanpercentile(values, [2.5, 97.5])
    return lo, hi


table = []
for key, label, higher in ROWS:
    a = np.array([b[key] for b in boots[A]]); b = np.array([b_[key] for b_ in boots[B]])
    diff = b - a
    p_val = min(1.0, 2 * min(np.nanmean(diff <= 0), np.nanmean(diff >= 0)))
    d_point = point[B][key] - point[A][key]
    better = "" if np.isnan(d_point) or p_val >= 0.05 else (B if (d_point > 0) == higher else A)
    table.append({"metric": label, "key": key,
                  A: point[A][key], f"{A} ci95": "[%.3f, %.3f]" % ci(a),
                  B: point[B][key], f"{B} ci95": "[%.3f, %.3f]" % ci(b),
                  "diff (B-A)": d_point, "diff ci95": "[%+.3f, %+.3f]" % ci(diff), "p": p_val, "better (p<0.05)": better})
for k in ("params_M", "ms_per_image"):
    table.append({"metric": k, "key": k, A: meta[A][k], B: meta[B][k], "diff (B-A)": meta[B][k] - meta[A][k]})
for level in ("image", "leaf"):
    table.append({"metric": f"{level} missed leaf (images)", "key": f"{level}_missed_leaf",
                  A: point[A][f"{level}_missed_leaf"], B: point[B][f"{level}_missed_leaf"]})

out_csv = RESULTS_DIR / f"{A}_vs_{B}_{DATA_NAME}.csv"
with open(out_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(table[0]))
    w.writeheader(); w.writerows(table)
print("written", out_csv.relative_to(ROOT), "\n")

fmt = lambda v: f"{v:8.3f}" if isinstance(v, float) else f"{v!s:>8}"
print(f"{'metric':<44}{A:>16}{B:>16}{'B − A':>10}{'p':>8}  better")
for r in table:
    print(f"{r['metric']:<44}{fmt(r[A]):>16}{fmt(r[B]):>16}{fmt(r.get('diff (B-A)', '')):>10}"
          f"{fmt(r.get('p', '')):>8}  {r.get('better (p<0.05)', '')}")

## Figures

Saved to `results/`: per-class IoU with 95 % CIs, predicted vs true coverage, Bland–Altman, and example predictions.

In [ ]:
colors = {A: "#4c72b0", B: "#dd8452"}

# Per-class IoU with bootstrap CIs
fig, ax = plt.subplots(figsize=(7, 4))
keys = ["miou_fg", *[f"iou_{c}" for c in CLASSES[1:]]]
x = np.arange(len(keys))
for j, m in enumerate(MODELS):
    vals = [point[m][k] for k in keys]
    lo_hi = np.array([ci([b[k] for b in boots[m]]) for k in keys])
    err = np.abs(lo_hi.T - np.array(vals))
    ax.bar(x + (j - 0.5) * 0.38, vals, 0.38, yerr=err, capsize=3, label=m, color=colors[m])
ax.set_xticks(x); ax.set_xticklabels(["mIoU", *CLASSES[1:]]); ax.set_ylabel("IoU (test)"); ax.set_ylim(0, 1); ax.legend()
fig.savefig(RESULTS_DIR / f"vs_iou_{DATA_NAME}.png", dpi=150, bbox_inches="tight")
plt.show()

# Predicted vs true image-level coverage, and Bland–Altman
fig, axes = plt.subplots(2, len(DISEASES), figsize=(6 * len(DISEASES), 10))
for col, d in enumerate(DISEASES):
    for m in MODELS:
        pairs = [(g["image"][d], r["image"][d]) for g, r in zip(gt, per_image[m]) if g["image"] and r["image"]]
        g_, p_ = np.array(pairs).T
        axes[0][col].scatter(g_, p_, s=14, alpha=0.7, color=colors[m], label=f"{m} (CCC {lins_ccc(g_, p_):.2f})")
        mean, diff = (g_ + p_) / 2, p_ - g_
        bias, loa = diff.mean(), 1.96 * diff.std(ddof=1)
        axes[1][col].scatter(mean, diff, s=14, alpha=0.7, color=colors[m], label=f"{m}: bias {bias:+.2f}, LoA ±{loa:.2f} pp")
        for y, style in ((bias, "-"), (bias + loa, "--"), (bias - loa, "--")):
            axes[1][col].axhline(y, color=colors[m], linestyle=style, linewidth=1)
    hi = max(axes[0][col].get_xlim()[1], axes[0][col].get_ylim()[1])
    axes[0][col].plot([0, hi], [0, hi], "k:", linewidth=1)
    axes[0][col].set(title=f"{d}: image-level coverage", xlabel="true (%)", ylabel="predicted (%)")
    axes[1][col].set(title=f"{d}: Bland–Altman", xlabel="mean of true and predicted (%)", ylabel="predicted − true (pp)")
    axes[0][col].legend(fontsize=8); axes[1][col].legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"vs_coverage_{DATA_NAME}.png", dpi=150, bbox_inches="tight")
plt.show()

# Example predictions
PALETTE = np.array([(0, 0, 0), (0, 200, 0), (230, 120, 0), (180, 0, 220)], np.uint8)  # bg, leaf, spot, mildew
fig, axes = plt.subplots(len(EXAMPLES), 4, figsize=(16, 4 * len(EXAMPLES)), squeeze=False)
for row, k in zip(axes, EXAMPLES):
    row[0].imshow(Image.open(TEST_DIR / images[k]["file_name"]).convert("RGB")); row[0].set_title("image")
    cov = gt[k]["image"] or {d: 0 for d in DISEASES}
    row[1].imshow(PALETTE[gt[k]["map"]]); row[1].set_title(f"ground truth (spot {cov['leaf_spot']:.1f}%, mildew {cov['powdery_mildew']:.1f}%)", fontsize=9)
    for j, m in enumerate(MODELS):
        pc = per_image[m][k]["image"] or {d: 0 for d in DISEASES}
        row[2 + j].imshow(PALETTE[examples[m][k]])
        row[2 + j].set_title(f"{m} (spot {pc['leaf_spot']:.1f}%, mildew {pc['powdery_mildew']:.1f}%)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("green = leaf, orange = leaf spot, purple = powdery mildew")
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"vs_examples_{DATA_NAME}.png", dpi=120, bbox_inches="tight")
plt.show()

## Reading the results

- **Image-level** metrics are the fair comparison: both models are scored on the same per-pixel class map.
- **Leaf-level** (largest leaf) is unstable on multi-leaf photos: a small change in which leaf is "largest" changes the number completely. The per-pixel model merges touching leaves into one big region, so its "largest leaf" is close to the whole image; YOLO picks one real leaf, often a different one than the ground truth. Use leaf-level only on single-leaf photos (the app's use case).
- Look at the example figure before trusting the mildew numbers: if the ground truth labels whole infected leaves as mildew while the models predict patches, the low mildew IoU is partly a labelling-convention problem.
- Check how many test images contain each disease (printed above). With few positives (e.g. powdery mildew), a difference can be large and still not significant; report the CI, not only the point estimate.
- The comparison is only fair if both models are fully trained: compare the epoch counts printed in *The two models* (YOLO was planned for 150 epochs).
- `p < 0.05` is per metric, without correction for the number of metrics; for the paper, pick the primary metrics in advance (e.g. image-level mIoU and coverage MAE per disease).